## Basic Usage
Simple Workflow
A straightforward sequence of operations demonstrating basic workflow structure:

In [1]:
from pydantic import BaseModel
from beeai_framework.workflows.workflow import Workflow

class State(BaseModel):
    message: str

async def first_step(state: State):
    state.message += " BeeAI"
    print("First Step executed")
    return Workflow.NEXT

async def second_step(state: State):
    state.message += " Framework"
    print("Second Step executed")
    return Workflow.END

workflow = Workflow(schema=State)
workflow.add_step("first_step", first_step)
workflow.add_step("second_step", second_step)

response = await workflow.run(State(message="Hello"))
print(response.state.message)

First Step executed
Second Step executed
Hello BeeAI Framework


### Multi-Step Workflow
Demonstrating conditional logic and loops within workflows, this example implements multiplication using repeated addition:

In [3]:
from pydantic import BaseModel
from beeai_framework.workflows.workflow import Workflow

class CalcState(BaseModel):
    x: int
    y: int
    result: int = 0

async def multiply(state: CalcState):
    if state.y > 0:
        for _ in range(state.y):
            state.result += state.x
    elif state.y < 0:
        for _ in range(abs(state.y)):
            state.result -= state.x
    return Workflow.END

calc_workflow = Workflow(schema=CalcState)
calc_workflow.add_step("multiply", multiply)

response = await calc_workflow.run(CalcState(x=7, y=-3))
print(response.state.result)

-21


## Advanced Orchestration

### Multi-Agent Workflows

BeeAI excels at orchestrating multi-agent systems, allowing you to integrate specialized agents to tackle complex tasks collaboratively.  An `AgentWorkflow` is specifically designed to manage and coordinate multiple `BeeAgent` instances, enabling the creation of sophisticated, modular agent-based applications. This section will guide you through building such a system.

In [1]:
import asyncio
import traceback
from pydantic import ValidationError
from beeai_framework.agents.bee.agent import BeeAgentExecutionConfig
from beeai_framework.backend.chat import ChatModel
from beeai_framework.backend.message import UserMessage
from beeai_framework.memory import UnconstrainedMemory
from beeai_framework.tools.search.duckduckgo import DuckDuckGoSearchTool
from beeai_framework.tools.weather.openmeteo import OpenMeteoTool
from beeai_framework.workflows.agent import AgentFactoryInput, AgentWorkflow
from beeai_framework.workflows.workflow import WorkflowError

async def run_workflow(prompt):
    llm = await ChatModel.from_name("ollama:granite3.1-dense:8b")

    try:
        workflow = AgentWorkflow(name="Smart assistant")
        workflow.add_agent(
            agent=AgentFactoryInput(
                name="WeatherForecaster",
                instructions="You are a weather assistant. Respond only if you can provide a useful answer.",
                tools=[OpenMeteoTool()],
                llm=llm,
                execution=BeeAgentExecutionConfig(max_iterations=3),
            )
        )
        workflow.add_agent(
            agent=AgentFactoryInput(
                name="Researcher",
                instructions="You are a researcher assistant. Respond only if you can provide a useful answer.",
                tools=[DuckDuckGoSearchTool()],
                llm=llm,
            )
        )
        workflow.add_agent(
            agent=AgentFactoryInput(
                name="Solver",
                instructions="""Your task is to provide the most useful final answer based on the assistants'
responses which all are relevant. Ignore those where assistant do not know.""",
                llm=llm,
            )
        )

        memory = UnconstrainedMemory()
        await memory.add(UserMessage(content=prompt))
        response = await workflow.run(messages=memory.messages)
        return response.state.final_answer

    except WorkflowError:
        traceback.print_exc()
        return None
    except ValidationError:
        traceback.print_exc()
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        traceback.print_exc()
        return None

async def execute_in_notebook(prompt):
    """Executes the workflow in a Jupyter Notebook environment."""
    try:
        result = await run_workflow(prompt)
        if result:
            print(f"result: {result}")
        else:
            print("Workflow execution failed.")
    except RuntimeError as e:
        if "cannot be called from a running event loop" in str(e):
            print("Error: asyncio.run() cannot be called from a running event loop in Jupyter. Use await directly.")
            print("Try using: await execute_in_notebook('Your Prompt Here')")
        else:
            print(f"An unexpected RuntimeError occurred: {e}")
            traceback.print_exc()
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        traceback.print_exc()

def execute_in_normal_python(prompt):
    """Executes the workflow in a normal Python environment."""
    try:
        result = asyncio.run(run_workflow(prompt))
        if result:
            print(f"result: {result}")
        else:
            print("Workflow execution failed.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        traceback.print_exc()



c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\pydantic\_internal\_fields.py:192: UserWarning: Field name "schema" in "ChatModelStructureInput" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [2]:
# Example usage within a Jupyter Notebook cell:
prompt = "What is the weather in Genova Italy?"
await execute_in_notebook(prompt)


result: The current weather in Genova, Italy is 22.5°C with partly cloudy skies and a relative humidity of 88%. Today's high is expected to be 30.5°C with no precipitation. Tomorrow's forecast calls for a slight chance of rain with a high of 28°C.


In [1]:
import asyncio
import traceback
from pydantic import ValidationError
from beeai_framework.agents.bee.agent import BeeAgentExecutionConfig
from beeai_framework.backend.chat import ChatModel
from beeai_framework.backend.message import UserMessage
from beeai_framework.memory import UnconstrainedMemory
from beeai_framework.tools.search.duckduckgo import DuckDuckGoSearchTool
from beeai_framework.tools.weather.openmeteo import OpenMeteoTool
from beeai_framework.workflows.agent import AgentFactoryInput, AgentWorkflow
from beeai_framework.workflows.workflow import WorkflowError

async def run_workflow(prompt):
    llm = await ChatModel.from_name("ollama:granite3.1-dense:8b")

    try:
        workflow = AgentWorkflow(name="Smart assistant")
        workflow.add_agent(
            agent=AgentFactoryInput(
                name="WeatherForecaster",
                instructions="You are a weather assistant. Respond only if you can provide a useful answer.",
                tools=[OpenMeteoTool()],
                llm=llm,
                execution=BeeAgentExecutionConfig(max_iterations=3),
            )
        )
        workflow.add_agent(
            agent=AgentFactoryInput(
                name="Researcher",
                instructions="You are a researcher assistant. Respond only if you can provide a useful answer.",
                tools=[DuckDuckGoSearchTool()],
                llm=llm,
            )
        )
        workflow.add_agent(
            agent=AgentFactoryInput(
                name="Solver",
                instructions="""Your task is to provide the most useful final answer based on the assistants'
responses which all are relevant. Ignore those where assistant do not know.""",
                llm=llm,
            )
        )

        memory = UnconstrainedMemory()
        await memory.add(UserMessage(content=prompt))
        response = await workflow.run(messages=memory.messages)
        return response.state.final_answer

    except WorkflowError:
        traceback.print_exc()
        return None
    except ValidationError:
        traceback.print_exc()
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        traceback.print_exc()
        return None

async def execute_in_notebook(prompt):
    """Executes the workflow in a Jupyter Notebook environment."""
    try:
        result = await run_workflow(prompt)
        if result:
            print(f"result: {result}")
        else:
            print("Workflow execution failed.")
    except RuntimeError as e:
        if "cannot be called from a running event loop" in str(e):
            print("Error: asyncio.run() cannot be called from a running event loop in Jupyter. Use await directly.")
            print("Try using: await execute_in_notebook('Your Prompt Here')")
        else:
            print(f"An unexpected RuntimeError occurred: {e}")
            traceback.print_exc()
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        traceback.print_exc()

# Example usage within a Jupyter Notebook cell:

c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\pydantic\_internal\_fields.py:192: UserWarning: Field name "schema" in "ChatModelStructureInput" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [2]:
# Example usage within a Jupyter Notebook cell:
prompt = "What is the weather in Genova Italy?"
await execute_in_notebook(prompt)


result: The current weather in Genova, Italy is 22.5°C with no rain and a wind speed of 4.1 km/h.


### Orchestrating with Ollama

In [ ]:
from beeai_framework.adapters.ollama.backend.chat import OllamaChatModel
from beeai_framework.backend.message import UserMessage

async def ollama_example():
    # Use a valid model name; if you intend to use "llama3.1", make sure to pull it first.
    llm = OllamaChatModel("granite3.1-dense:8b")
    user_message = UserMessage("What is the capital of Italy?")
    
    try:
        # Pass messages as a dictionary with the key "messages"
        response = await llm.create({"messages": [user_message]})
        print(response.get_text_content())
    except Exception as e:
        print("An error occurred:", e)
        print("Ensure the model name is correct or pull the model if necessary.")

# In an async-capable environment (e.g., Jupyter, or within an async main), run:
await ollama_example()


c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\pydantic\_internal\_fields.py:192: UserWarning: Field name "schema" in "ChatModelStructureInput" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [8]:
import os
from dotenv import load_dotenv
from beeai_framework.backend.chat import ChatModel, ChatModelOutput
from beeai_framework.backend.message import UserMessage

# Load environment variables from .env file
load_dotenv()

# Retrieve environment variables
WATSONX_PROJECT_ID = os.getenv("PROJECT_ID")
WATSONX_API_KEY = os.getenv("WATSONX_API_KEY")
WATSONX_API_URL = os.getenv("WATSONX_URL")

# Await the creation of the ChatModel instance
model = await ChatModel.from_name(
    "watsonx:ibm/granite-3-8b-instruct",
    options={
        "project_id": WATSONX_PROJECT_ID,
        "api_key": WATSONX_API_KEY,
        "api_base": WATSONX_API_URL,
    },
)

message = UserMessage(content="Briefly explain quantum computing in simple terms with an example.")
# Pass the messages as a dictionary.
output: ChatModelOutput = await model.create({"messages": [message]})

print(output.get_text_content())


Quantum computing is a type of computing that uses quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data. Unlike classical computers that use bits (0s and 1s) for processing, quantum computers use quantum bits, or qubits.

A classical bit can be in one of two states - 0 or 1. However, due to a principle called superposition, a qubit can be in multiple states at once - it can be 0, 1, or both at the same time, represented as |0⟩, |1⟩, or (|0⟩ + |1⟩)/√2. This allows quantum computers to effectively process a vast number of possibilities all at once.

Another key principle is entanglement, where two or more qubits become connected in such a way that the state of one (whether it's 0 or 1, or both) can instantaneously affect the state of another, regardless of the physical distance between them. This interconnectedness further enhances the computational power.

Let's consider an example to illustrate this power. Suppose we want to search through

In [10]:
import os
from dotenv import load_dotenv
from beeai_framework.adapters.watsonx.backend.chat import WatsonxChatModel
from beeai_framework.backend.message import UserMessage

# Load environment variables from .env
load_dotenv()

WATSONX_PROJECT_ID = os.getenv("PROJECT_ID")
WATSONX_API_KEY = os.getenv("WATSONX_API_KEY")
WATSONX_API_URL = os.getenv("WATSONX_URL")

async def watsonx_example():
    # Await model instantiation with correct provider-prefixed model name and options
    watsonx_llm = await WatsonxChatModel.from_name(
        "watsonx:ibm/granite-3-8b-instruct",
        options={
            "project_id": WATSONX_PROJECT_ID,
            "api_key": WATSONX_API_KEY,
            "api_base": WATSONX_API_URL,
        },
    )
    user_message = UserMessage("Historical significance of the Eiffel Tower?")
    # Pass the messages inside a dictionary with the key "messages"
    response = await watsonx_llm.create({"messages": [user_message]})
    print(response.get_text_content())

await watsonx_example()


The Eiffel Tower, located in Paris, France, is one of the most iconic structures in the world and carries immense historical significance. 

1. Construction (1887-1889): Designed by Gustave Eiffel for the 1889 Exposition Universelle (World's Fair), commemorating the 100th year anniversary of the French Revolution. Its original purpose was to serve as the entrance arch for the exhibition and to demonstrate France's industrial prowess.

2. Technology and Engineering: At the time of its completion, the Eiffel Tower was the tallest man-made structure in the world, standing at 324 meters (1,063 feet). It showcased advanced engineering techniques and became a representation of structural innovation.

3. Symbol of French Culture and Industry: The Eiffel Tower became an instant cultural icon for France, reflecting the country's industrial might and architectural ambition. It transformed Paris, establishing a new architectural style that coexisted amid the city's historic landmarks.

4. Scienti

In [ ]:
import asyncio
import sys
import traceback
import os
from dotenv import load_dotenv

# Optional: Apply nest_asyncio to allow asyncio.run() in environments with a running loop (e.g., Jupyter)
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

from beeai_framework.agents.types import BeeAgentExecutionConfig
from beeai_framework.workflows.agent import AgentFactoryInput, AgentWorkflow
from beeai_framework.adapters.watsonx.backend.chat import WatsonxChatModel
from beeai_framework.adapters.ollama.backend.chat import OllamaChatModel
from beeai_framework.backend.message import UserMessage
from beeai_framework.errors import FrameworkError
from beeai_framework.memory import UnconstrainedMemory
from beeai_framework.tools.search.duckduckgo import DuckDuckGoSearchTool
from beeai_framework.tools.weather.openmeteo import OpenMeteoTool

# Load environment variables from .env
load_dotenv()
WATSONX_PROJECT_ID = os.getenv("PROJECT_ID")
WATSONX_API_KEY = os.getenv("WATSONX_API_KEY")
WATSONX_API_URL = os.getenv("WATSONX_URL")

async def test_model(llm, query: str) -> str:
    """
    Test a given language model by sending a query and returning the response text.
    """
    user_message = UserMessage(content=query)
    try:
        response = await llm.create({"messages": [user_message]})
        result_text = response.get_text_content()
        print(f"Response from model {llm.__class__.__name__}: {result_text}")
        return result_text
    except Exception as e:
        print(f"Error testing model {llm.__class__.__name__}: {e}")
        return ""

async def run_mixed_workflow() -> None:
    """
    Set up and run the mixed-backend workflow using Watsonx and Ollama.
    """
    # Initialize Watsonx ChatModel using environment variables
    llm_watsonx = await WatsonxChatModel.from_name(
        "watsonx:ibm/granite-3-8b-instruct",
        options={
            "project_id": WATSONX_PROJECT_ID,
            "api_key": WATSONX_API_KEY,
            "api_base": WATSONX_API_URL,
        },
    )
    # Initialize Ollama ChatModel (no env options required)
    llm_ollama = OllamaChatModel("granite3.1-dense:8b")

    workflow = AgentWorkflow(name="Smart assistant (Mixed Backend)")
    
    # Add WeatherForecaster agent using Watsonx
    workflow.add_agent(
        AgentFactoryInput(
            name="WeatherForecaster",
            instructions="You are a weather assistant.",
            tools=[OpenMeteoTool()],
            llm=llm_watsonx,
            execution=BeeAgentExecutionConfig(
                max_iterations=3, total_max_retries=10, max_retries_per_step=3
            ),
        )
    )
    
    # Add Researcher agent using Ollama
    workflow.add_agent(
        AgentFactoryInput(
            name="Researcher",
            instructions="You are a researcher assistant.",
            tools=[DuckDuckGoSearchTool()],
            llm=llm_ollama,
        )
    )
    
    # Add Solver agent using Watsonx
    workflow.add_agent(
        AgentFactoryInput(
            name="Solver",
            instructions=(
                "Your task is to provide the most useful final answer based on the assistants' "
                "responses which all are relevant. Ignore those where assistant do not know."
            ),
            llm=llm_watsonx,
        )
    )

    prompt = "What is the weather in London and the capital of France?"
    memory = UnconstrainedMemory()
    await memory.add(UserMessage(content=prompt))
    
    response = await workflow.run(messages=memory.messages)
    print(f"Result from mixed workflow: {response.state.final_answer}")

async def main() -> None:
    # Test Watsonx model
    print("Testing Watsonx model:")
    llm_watsonx = await WatsonxChatModel.from_name(
        "watsonx:ibm/granite-3-8b-instruct",
        options={
            "project_id": WATSONX_PROJECT_ID,
            "api_key": WATSONX_API_KEY,
            "api_base": WATSONX_API_URL,
        },
    )
    await test_model(llm_watsonx, "Describe the Eiffel Tower.")

    # Test Ollama model
    print("\nTesting Ollama model:")
    llm_ollama = OllamaChatModel("granite3.1-dense:8b")
    await test_model(llm_ollama, "Describe the Eiffel Tower.")

    # Run the mixed backend workflow
    print("\nRunning mixed backend workflow:")
    await run_mixed_workflow()

if __name__ == "__main__":
    try:
        # This allows the code to run in environments (like Jupyter) with an already running event loop.
        asyncio.run(main())
    except FrameworkError as e:
        traceback.print_exc()
        sys.exit(e.explain())
    except Exception as e:
        traceback.print_exc()
        sys.exit(str(e))


c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\pydantic\_internal\_fields.py:192: UserWarning: Field name "schema" in "ChatModelStructureInput" shadows an attribute in parent "BaseModel"
  warnings.warn(


Testing Watsonx model:
Response from model WatsonxChatModel: The Eiffel Tower, located in Paris, France, is a wrought-iron lattice tower that was constructed from 1887 to 1889 as the entrance arch for the 1889 World's Fair. Named after its engineer, Gustave Eiffel, it stood as the tallest man-made structure in the world until the completion of the Chrysler Building in New York in 1930. Standing at 324 meters (1,063 feet) high, including antennas, it remains one of the most recognizable landmarks globally.

The tower is divided into three levels for tourists, with restaurants on the first and second levels offering panoramic views of Paris. The top level hosts an antenna, and the Sparkling Eiffel Tower light show occurs every hour on the hour after sunset.

The design of the tower is characterized by its distinct three-paneled structure, with curved sides on the first and second levels and straight sides on the top level. Each of the tower's four legs has a slight outward cant (tilt), w

In [1]:
import asyncio
import sys
import traceback
import os
from dotenv import load_dotenv

# Optional: Apply nest_asyncio to allow asyncio.run() in environments with a running loop (e.g., Jupyter)
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

from beeai_framework.agents.types import BeeAgentExecutionConfig
from beeai_framework.workflows.agent import AgentFactoryInput, AgentWorkflow
from beeai_framework.adapters.watsonx.backend.chat import WatsonxChatModel
from beeai_framework.adapters.ollama.backend.chat import OllamaChatModel
from beeai_framework.backend.message import UserMessage
from beeai_framework.errors import FrameworkError
from beeai_framework.memory import UnconstrainedMemory
from beeai_framework.tools.search.duckduckgo import DuckDuckGoSearchTool
from beeai_framework.tools.weather.openmeteo import OpenMeteoTool

# Load environment variables from .env
load_dotenv()
WATSONX_PROJECT_ID = os.getenv("PROJECT_ID")
WATSONX_API_KEY = os.getenv("WATSONX_API_KEY")
WATSONX_API_URL = os.getenv("WATSONX_URL")

async def test_model(llm, query: str) -> str:
    """
    Test a given language model by sending a query and returning the response text.
    Note: Using a positional argument for UserMessage to mimic the working example.
    """
    # Use a positional argument as in the working example for Ollama
    user_message = UserMessage(query)
    try:
        # Add a timeout to avoid indefinite hanging (adjust timeout as needed)
        response = await asyncio.wait_for(llm.create({"messages": [user_message]}), timeout=30)
        result_text = response.get_text_content()
        print(f"Response from model {llm.__class__.__name__}: {result_text}")
        return result_text
    except Exception as e:
        print(f"Error testing model {llm.__class__.__name__}: {e}")
        return ""

async def run_mixed_workflow() -> None:
    """
    Set up and run the mixed-backend workflow using Watsonx and Ollama.
    """
    # Initialize Watsonx ChatModel using environment variables
    llm_watsonx = await WatsonxChatModel.from_name(
        "watsonx:ibm/granite-3-8b-instruct",
        options={
            "project_id": WATSONX_PROJECT_ID,
            "api_key": WATSONX_API_KEY,
            "api_base": WATSONX_API_URL,
        },
    )
    # Initialize Ollama ChatModel (no env options required)
    llm_ollama = OllamaChatModel("granite3.1-dense:8b")

    workflow = AgentWorkflow(name="Smart assistant (Mixed Backend)")
    
    # Add WeatherForecaster agent using Watsonx
    workflow.add_agent(
        AgentFactoryInput(
            name="WeatherForecaster",
            instructions="You are a weather assistant.",
            tools=[OpenMeteoTool()],
            llm=llm_watsonx,
            execution=BeeAgentExecutionConfig(
                max_iterations=3, total_max_retries=10, max_retries_per_step=3
            ),
        )
    )
    
    # Add Researcher agent using Ollama
    workflow.add_agent(
        AgentFactoryInput(
            name="Researcher",
            instructions="You are a researcher assistant.",
            tools=[DuckDuckGoSearchTool()],
            llm=llm_ollama,
        )
    )
    
    # Add Solver agent using Watsonx
    workflow.add_agent(
        AgentFactoryInput(
            name="Solver",
            instructions=(
                "Your task is to provide the most useful final answer based on the assistants' "
                "responses which all are relevant. Ignore those where assistant do not know."
            ),
            llm=llm_watsonx,
        )
    )

    prompt = "What is the weather in London and the capital of France?"
    memory = UnconstrainedMemory()
    await memory.add(UserMessage(prompt))
    
    response = await workflow.run(messages=memory.messages)
    print(f"Result from mixed workflow: {response.state.final_answer}")

async def main() -> None:
    # Test Ollama model first
    print("Testing Ollama model:")
    llm_ollama = OllamaChatModel("granite3.1-dense:8b")
    await test_model(llm_ollama, "What is the capital of Italy?")

    # Then test Watsonx model
    print("\nTesting Watsonx model:")
    llm_watsonx = await WatsonxChatModel.from_name(
        "watsonx:ibm/granite-3-8b-instruct",
        options={
            "project_id": WATSONX_PROJECT_ID,
            "api_key": WATSONX_API_KEY,
            "api_base": WATSONX_API_URL,
        },
    )
    await test_model(llm_watsonx, "What is the capital of Italy?")

    # Run the mixed backend workflow
    print("\nRunning mixed backend workflow:")
    await run_mixed_workflow()

if __name__ == "__main__":
    try:
        asyncio.run(main())
    except FrameworkError as e:
        traceback.print_exc()
        sys.exit(e.explain())
    except Exception as e:
        traceback.print_exc()
        sys.exit(str(e))


c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\pydantic\_internal\_fields.py:192: UserWarning: Field name "schema" in "ChatModelStructureInput" shadows an attribute in parent "BaseModel"
  warnings.warn(


Testing Ollama model:

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error testing model OllamaChatModel: unhashable type: 'ChatModelInput'

Testing Watsonx model:
Response from model WatsonxChatModel: The capital of Italy is Rome. Established as the center of the Roman Empire in the 1st century BCE, Rome is renowned for its ancient architecture, including structures like the Colosseum and Roman Forum. It also houses iconic locations such as the Vatican City, an independent city-state enclaved within Rome, acting as the spiritual and administrative headquarters of the Roman Catholic Church. Today, Rome is a vibrant, bustling city known for its rich history, art, architecture, and being one of Europe's most famous tourist destinations.

If you have any other questions or need information on a different topic, please feel free to ask!

Running mixed backend workflow:

Give Feedback /

Traceback (most recent call last):
  File "c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\litellm\main.py", line 469, in acompletion
    response = await init_response
               ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\litellm\llms\openai_like\chat\handler.py", line 138, in acompletion_stream_function
    completion_stream = await make_call(
                        ^^^^^^^^^^^^^^^^
  File "c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\litellm\llms\openai_like\chat\handler.py", line 40, in make_call
    response = await client.post(
               ^^^^^^^^^^^^^^^^^^
  File "c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 131, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rusla\.conda\envs\beeai\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 236, in post
    raise e
  File "c:\Users\rusla\.conda

AttributeError: 'tuple' object has no attribute 'tb_frame'